[Python, Visually](https://johnfisher-ai.github.io/Python-Visual-Guides/) &nbsp;&rsaquo;&nbsp; [Testing and Packaging](https://johnfisher-ai.github.io/Python-Visual-Guides/testing-and-packaging.html)

# Project Layout


## What you will be able to do

Lay out a project so that its code imports from any folder: a package in `src`, tests in `tests`,
and a `pyproject.toml` that names the project, its version and its dependencies. Install the project
into its environment in editable mode, so that an edit takes effect without installing again, run
its tests from anywhere, give it a command of its own, and build a wheel that somebody else can
install.


## The idea

### The problem

The project in this guide has been a folder: `readings.py`, a script, and a `tests` folder beside
them. Its tests import `readings` for one reason, which the **Your First Test** notebook named:
`python -m pytest`, run in the project's folder, puts that folder on the import path. The
**Virtual Environments** notebook ran the environment's own `pytest` script, which does not, and the
tests could not find the module. The **Why Test** notebook put the folder on the import path by hand.

Everything about the project depends on where a command runs. Nothing says that the folder is a
package called `stations`, at version 0.1.0, that needs Python 3.10 or later, so it cannot be
installed into an environment, and another project cannot depend on it. And a test run in the
project's folder may import the files lying there, rather than what an install would give a user,
and pass for code that nobody else can import.

### What a project layout is

> A **project layout** is where a project keeps its files. In the **src layout**, the importable
> package sits in a folder of its own under `src`, such as `src/stations/`, the tests sit in
> `tests/`, and the project is described by **`pyproject.toml`**. That file names the **build
> backend**, the tool that turns the folder into something pip can install, such as setuptools, and
> the project's metadata: its name, its version, the Pythons it supports, and its dependencies.
> `pip install .` builds the project and installs a copy of the package into the environment. An
> **editable install**, `pip install -e .`, installs a link to the project's folder instead, so that
> an edit takes effect without installing again. Either way, `import stations` works from any
> folder, because the package is installed, and not because of where the command ran.

### Why it works that way

- **An import searches the import path, not the folder you stand in.** `python -m` adds the folder
  it runs in to the path, a script such as `pytest` does not, and neither helps when the command
  runs somewhere else. An installed package is on the path of its environment's Python, wherever it
  runs.
- **`src` keeps the uninstalled copy out of reach.** A package at the top of the project is
  importable from the project's folder whether it installs or not. Under `src` it is not, so a test
  imports what the install put in place.
- **`pyproject.toml` is the project's own record.** pip reads it to learn how to build the project
  and what the project needs, and other tools, pytest among them, keep their settings in it.
- **pip builds in isolation.** It installs the build backend that `[build-system]` names into a
  temporary environment of its own, so the project's environment does not need setuptools.
- **Editable for working on a project, regular for using one.** An editable install follows the
  project's files as they change. A regular install is a copy, which is what a user receives.
- **A command is a function with a name.** `[project.scripts]` makes pip write a script into the
  environment's `bin` folder that calls one function of the package.

### Where this shows up

The Python Packaging Authority's tutorial, Packaging Python Projects, lays out its example in exactly
this way, with a src layout and `pyproject.toml`, and its guide to the two layouts gives the reasons
above. pytest's guide to good integration practices suggests keeping tests outside the package, in a
`tests` folder, and setuptools finds a package under `src` without being told. The **Requirements and
Pinning** notebook pinned the versions of an environment. A project states its dependencies in
`pyproject.toml` as ranges, and the **Continuous Integration** notebook installs this project with
`pip install -e .` on every push.

### What this notebook covers

- A flat project, and the one folder its tests work from
- The src layout, and a package that imports only once it is installed
- `pyproject.toml`, read as a record of the project
- An editable install, and imports that work from any folder
- Tests that import the installed package, however they are run
- An edit that takes effect without installing again
- A command of its own, from `[project.scripts]`
- A wheel, and a regular install of it
- The project rebuilt from its files in another folder, with a `.gitignore`
- Four errors: a flat project with two modules, a Python the project was never installed into, an
  import written for the flat layout, and a regular install that does not see an edit

### A first look

Before any of the detail, here is the whole idea in a few lines. There is nothing to run yet: read
it, and read the output underneath it. Everything from Setup onward is where you start running
things, and the rest of the notebook takes this apart piece by piece.

```python
import subprocess
import sys
import tempfile
from pathlib import Path

project = Path(tempfile.mkdtemp())
(project / "src" / "stations").mkdir(parents=True)
(project / "src" / "stations" / "__init__.py").write_text("")
(project / "src" / "stations" / "readings.py").write_text(
    "def to_fahrenheit(celsius):\n    return celsius * 9 / 5 + 32\n")
(project / "pyproject.toml").write_text("""[build-system]
requires = ["setuptools>=77.0.3"]
build-backend = "setuptools.build_meta"

[project]
name = "stations"
version = "0.1.0"
""")

environment = project / ".venv"
subprocess.run([sys.executable, "-m", "venv", "--without-pip", environment], check=True)
subprocess.run([sys.executable, "-m", "pip", "--python", environment, "--disable-pip-version-check",
                "install", "-q", "-e", project], check=True)

check = "from stations.readings import to_fahrenheit; print('100 C is', to_fahrenheit(100), 'F')"
subprocess.run([environment / "bin" / "python", "-c", check], cwd=tempfile.mkdtemp())
```

```
100 C is 212.0 F
```

A package in `src`, a `pyproject.toml` of seven lines, and an editable install into an environment.
The last command runs in a new, empty folder, far from the project, and `import stations` still
works: the package is installed, so the folder a command runs in no longer matters.


## Setup

Eight imports, and the two functions that run commands.

- `subprocess` runs `venv`, pip, pytest and the project's command as programs of their own
- `sys` names the notebook's own Python, which makes every environment and runs pip
- `os` sets `NO_COLOR`, `PYTHONDONTWRITEBYTECODE` and `COLUMNS` for those programs
- `re` takes the time a test run took out of pytest's report
- `tomllib` reads `pyproject.toml`
- `zipfile` lists the files inside a wheel
- `Path` names the project's folders, and writes and reads its files
- `shutil` copies the project, and removes the scratch folder at the end

`run` runs a command in a folder, `scratch` unless it is told otherwise, and returns its exit code
and what it printed, with that folder's full path and the time a test run took taken out. `pip` runs
the notebook's pip for the Python in an environment, as the **Virtual Environments** notebook
explained, with `--no-color`, for the reason the **Requirements and Pinning** notebook gave. The
project lives in `scratch/stations`, and a flat version of it in `scratch/flat`, so that a command
run in `scratch` runs outside both.


In [1]:
import os
import re
import shutil
import subprocess
import sys
import tomllib
import zipfile
from pathlib import Path

SCRATCH = Path("scratch")
PROJECT = SCRATCH / "stations"
FLAT = SCRATCH / "flat"
for folder in [PROJECT / "src" / "stations", PROJECT / "tests", FLAT / "tests"]:
    folder.mkdir(parents=True, exist_ok=True)
os.environ["NO_COLOR"] = "1"                  # programs started from here print without color codes
os.environ["PYTHONDONTWRITEBYTECODE"] = "1"   # and keep no compiled copies, which a quick rewrite can outrun
os.environ["COLUMNS"] = "80"                  # and print reports 80 characters wide


def run(*command, folder=SCRATCH):
    """Run a command in a folder, and return its exit code and what it printed, less this computer's paths."""
    finished = subprocess.run([str(part) for part in command], cwd=folder, capture_output=True, text=True)
    printed = (finished.stdout + finished.stderr).replace(f"{Path(folder).resolve()}/", "")
    return finished.returncode, re.sub(r" in \d+\.\d+s\b", "", printed).rstrip()


def pip(environment, *arguments, folder=SCRATCH):
    """Run this notebook's pip for the Python in an environment: python -m pip --python ENVIRONMENT ..."""
    return run(sys.executable, "-m", "pip", "--python", environment, "--disable-pip-version-check", "--no-color",
               *arguments, folder=folder)


print("ready:", PROJECT, "and", FLAT)


ready: scratch/stations and scratch/flat


## Worked examples

### A flat project, and the one folder its tests work from

First the project as it has been, flat: the module and a script at the top of its folder, and the
tests beside them:


In [2]:
%%writefile scratch/flat/readings.py
"""Readings from the weather stations, and each station's mean temperature."""

import statistics


def parse_reading(line):
    """A (station, celsius) pair from a line such as 'Bergen,4.2'. An empty reading is None."""
    station, celsius = line.strip().split(",")
    return station, float(celsius) if celsius else None


def mean(values):
    """The mean of the readings that are not None, or None when there are none."""
    present = [value for value in values if value is not None]
    return statistics.fmean(present) if present else None


def summarize(lines):
    """Each station's mean temperature, from lines of readings. A blank line is skipped."""
    by_station = {}
    for line in lines:
        if line.strip():
            station, celsius = parse_reading(line)
            by_station.setdefault(station, []).append(celsius)
    return {station: mean(values) for station, values in by_station.items()}


def to_fahrenheit(celsius):
    """A temperature in degrees Celsius, in degrees Fahrenheit."""
    return celsius * 9 / 5 + 32


Writing scratch/flat/readings.py


In [3]:
%%writefile scratch/flat/summary.py
"""Print each station's mean temperature from a file of readings: python summary.py readings.csv"""

import sys

from readings import summarize


def main(arguments=None):
    arguments = sys.argv[1:] if arguments is None else arguments
    with open(arguments[0], encoding="utf-8") as file:
        for station, celsius in summarize(file).items():
            print(station, "no readings" if celsius is None else f"{celsius:.1f}")


if __name__ == "__main__":
    main()


Writing scratch/flat/summary.py


In [4]:
%%writefile scratch/flat/tests/test_readings.py
from readings import mean, to_fahrenheit


def test_mean_of_two_readings():
    assert mean([4.2, 5.8]) == 5.0


def test_boiling_point_in_fahrenheit():
    assert to_fahrenheit(100) == 212


Writing scratch/flat/tests/test_readings.py


It gets an environment with pytest, and its tests run three ways: `python -m pytest` in the project's
folder, the environment's `pytest` script in the same folder, and `python -m pytest` from the folder
above. The cell prints each exit code with the last line of the report, or the `E` line of an error:


In [5]:
run(sys.executable, "-m", "venv", "--without-pip", "flat/.venv")
code, printed = pip("flat/.venv", "install", "-q", "pytest==8.4.2")
print("install exit code:", code)

ways = [("python -m pytest, in the project", [".venv/bin/python", "-m", "pytest", "-q", "--no-header"], FLAT),
        ("the pytest script, in the project", [".venv/bin/pytest", "-q", "--no-header"], FLAT),
        ("python -m pytest, from the folder above",
         ["flat/.venv/bin/python", "-m", "pytest", "-q", "--no-header", "flat"], SCRATCH)]
for name, command, folder in ways:
    code, printed = run(*command, folder=folder)
    shown = next((line for line in printed.splitlines() if line.startswith("E ")), printed.splitlines()[-1])
    print(f"{name:<40} exit code {code}: {shown.strip()}")


install exit code: 0
python -m pytest, in the project         exit code 0: 2 passed
the pytest script, in the project        exit code 2: E   ModuleNotFoundError: No module named 'readings'
python -m pytest, from the folder above  exit code 2: E   ModuleNotFoundError: No module named 'readings'


Only the first way works. `python -m` puts the folder it runs in on the import path, and
`readings.py` is in that folder. The `pytest` script does not put that folder on the path, and from
the folder above, `python -m` puts `scratch` there, where there is no `readings.py`. The tests
depend on where the command runs, and so would any program that imports `readings`.

### The src layout, and a package that imports only once it is installed

In the src layout, the code becomes a package, a folder with an `__init__.py`, inside `src`, and the
tests import it by its package name, `stations`:


In [6]:
%%writefile scratch/stations/src/stations/__init__.py
"""Mean temperatures from the weather stations' readings."""


Writing scratch/stations/src/stations/__init__.py


In [7]:
%%writefile scratch/stations/src/stations/readings.py
"""Readings from the weather stations, and each station's mean temperature."""

import statistics


def parse_reading(line):
    """A (station, celsius) pair from a line such as 'Bergen,4.2'. An empty reading is None."""
    station, celsius = line.strip().split(",")
    return station, float(celsius) if celsius else None


def mean(values):
    """The mean of the readings that are not None, or None when there are none."""
    present = [value for value in values if value is not None]
    return statistics.fmean(present) if present else None


def summarize(lines):
    """Each station's mean temperature, from lines of readings. A blank line is skipped."""
    by_station = {}
    for line in lines:
        if line.strip():
            station, celsius = parse_reading(line)
            by_station.setdefault(station, []).append(celsius)
    return {station: mean(values) for station, values in by_station.items()}


def to_fahrenheit(celsius):
    """A temperature in degrees Celsius, in degrees Fahrenheit."""
    return celsius * 9 / 5 + 32


Writing scratch/stations/src/stations/readings.py


In [8]:
%%writefile scratch/stations/src/stations/summary.py
"""Print each station's mean temperature from a file of readings: stations-summary readings.csv"""

import sys

from stations.readings import summarize, to_fahrenheit


def main(arguments=None):
    arguments = sys.argv[1:] if arguments is None else arguments
    with open(arguments[0], encoding="utf-8") as file:
        for station, celsius in summarize(file).items():
            if celsius is None:
                print(station, "no readings")
            else:
                print(station, f"{celsius:.1f} C, {to_fahrenheit(celsius):.1f} F")


Writing scratch/stations/src/stations/summary.py


In [9]:
%%writefile scratch/stations/tests/test_readings.py
from stations.readings import mean, summarize, to_fahrenheit


def test_mean_of_two_readings():
    assert mean([4.2, 5.8]) == 5.0


def test_boiling_point_in_fahrenheit():
    assert to_fahrenheit(100) == 212


def test_a_blank_line_is_skipped():
    assert summarize(["Bergen,4.2", "", "Bergen,5.8"]) == {"Bergen": 5.0}


Writing scratch/stations/tests/test_readings.py


In [10]:
print(sorted(str(path.relative_to(PROJECT)) for path in PROJECT.rglob("*") if path.is_file()))

run(sys.executable, "-m", "venv", "--without-pip", "stations/.venv")
pip(".venv", "install", "-q", "pytest==8.4.2", folder=PROJECT)
code, printed = run(".venv/bin/python", "-m", "pytest", "-q", "--no-header", folder=PROJECT)
print("exit code:", code)
print(next(line for line in printed.splitlines() if line.startswith("E ")).strip())


['src/stations/__init__.py', 'src/stations/readings.py', 'src/stations/summary.py', 'tests/test_readings.py']
exit code: 2
E   ModuleNotFoundError: No module named 'stations'


The test failed to import `stations` even with `python -m pytest` in the project's folder, which put
that folder on the import path: the package is in `src`, one folder down. That is the point of the
layout. The only way for anything to import `stations` is to install it, so the tests exercise the
package as it installs, and not files that happen to be lying in the folder.

### pyproject.toml, read as a record of the project

`pyproject.toml` is where the project says what it is. The file is TOML, a format of tables in square
brackets and `key = value` lines:


In [11]:
%%writefile scratch/stations/pyproject.toml
[build-system]
requires = ["setuptools>=77.0.3"]
build-backend = "setuptools.build_meta"

[project]
name = "stations"
version = "0.1.0"
description = "Mean temperatures from the weather stations' readings"
requires-python = ">=3.10"
dependencies = []

[project.optional-dependencies]
test = ["pytest==8.4.2"]

[project.scripts]
stations-summary = "stations.summary:main"

[tool.pytest.ini_options]
testpaths = ["tests"]


Writing scratch/stations/pyproject.toml


In [12]:
record = tomllib.loads((PROJECT / "pyproject.toml").read_text())
project = record["project"]

print("build backend:", record["build-system"]["build-backend"], "| needs", record["build-system"]["requires"])
print("project:", project["name"], project["version"], "| Python", project["requires-python"])
print("dependencies:", project["dependencies"], "| for testing:", project["optional-dependencies"]["test"])
print("commands:", project["scripts"])
print("pytest looks in:", record["tool"]["pytest"]["ini_options"]["testpaths"])


build backend: setuptools.build_meta | needs ['setuptools>=77.0.3']
project: stations 0.1.0 | Python >=3.10
dependencies: [] | for testing: ['pytest==8.4.2']
commands: {'stations-summary': 'stations.summary:main'}
pytest looks in: ['tests']


| Table | What it says |
|---|---|
| `[build-system]` | the build backend, setuptools here, and the versions of it pip may install to build the project |
| `[project]` | the name and version pip installs the project under, the Pythons it supports, and what it depends on |
| `[project.optional-dependencies]` | extras: named groups installed on request, such as `test`, with `pip install -e ".[test]"` |
| `[project.scripts]` | commands, each the name of a function to call |
| `[tool.pytest.ini_options]` | pytest's settings, here the folder it collects tests from when given none |

`dependencies` is empty, since the package imports only the standard library. A dependency goes there
as a range, such as `"packaging>=22"`, as the **Requirements and Pinning** notebook said a library
states its needs, and the pins belong in the requirements file of whatever runs it. `tomllib` reads
TOML and, since Python 3.11, comes with Python.

### An editable install, and imports that work from any folder

`pip install -e ".[test]"`, run in the project's folder, builds the project, installs it into the
environment in editable mode, and installs the `test` extra with it. pip downloads setuptools into a
temporary environment for the build, which takes a few seconds the first time:


In [13]:
code, printed = pip(".venv", "install", "-q", "-e", ".[test]", folder=PROJECT)
print("install exit code:", code)

check = "import stations.readings as readings; print(readings.__file__); print(readings.to_fahrenheit(100))"
code, printed = run("stations/.venv/bin/python", "-c", check, folder=SCRATCH)
print(printed)

listing = ("import os, sysconfig; "
           "print(sorted(name for name in os.listdir(sysconfig.get_paths()['purelib']) if 'stations' in name))")
print(run("stations/.venv/bin/python", "-c", listing)[1])


install exit code: 0
stations/src/stations/readings.py
212.0
['__editable__.stations-0.1.0.pth', 'stations-0.1.0.dist-info']


The import ran in `scratch`, outside the project, and found the module at
`stations/src/stations/readings.py`, the project's own file. The environment's `site-packages` holds
two entries for the project. `stations-0.1.0.dist-info` records that `stations` 0.1.0 is installed,
with its metadata, and `__editable__.stations-0.1.0.pth` holds one line, the path of the project's
`src` folder, which Python adds to the import path whenever the environment's Python starts. Nothing
was copied, so the environment reads the project's files wherever it runs.

### Tests that import the installed package

The tests now pass however they are run, since `stations` is installed, and `testpaths` tells pytest
where the tests are when it is given no folder:


In [14]:
ways = [("python -m pytest, in the project", [".venv/bin/python", "-m", "pytest", "-q", "--no-header"], PROJECT),
        ("the pytest script, in the project", [".venv/bin/pytest", "-q", "--no-header"], PROJECT),
        ("the pytest script, from the folder above",
         ["stations/.venv/bin/pytest", "-q", "--no-header", "stations"], SCRATCH)]
for name, command, folder in ways:
    code, printed = run(*command, folder=folder)
    print(f"{name:<41} exit code {code}: {printed.splitlines()[-1]}")


python -m pytest, in the project          exit code 0: 3 passed
the pytest script, in the project         exit code 0: 3 passed
the pytest script, from the folder above  exit code 0: 3 passed


Three ways, three passes, where the flat project passed one of them. The environment's Python finds
`stations` on its import path, wherever the command starts.

### An edit, without installing again

An editable install reads the project's files as they are, so an edit to the package reaches every
Python in the environment at once. Here `readings.py` gains a conversion to kelvin:


In [15]:
addition = ('\n\ndef to_kelvin(celsius):\n'
            '    """A temperature in degrees Celsius, in kelvin."""\n'
            '    return celsius + 273.15\n')
readings = PROJECT / "src" / "stations" / "readings.py"
readings.write_text(readings.read_text() + addition)

code, printed = run("stations/.venv/bin/python", "-c", "from stations.readings import to_kelvin; print(to_kelvin(100))")
print("exit code:", code, "|", printed)


exit code: 0 | 373.15


No install between the edit and the import. `pip install -e .` is needed again only when
`pyproject.toml` changes, such as a new dependency or a new command, since those are read at install
time.

### A command of its own

`[project.scripts]` named a command, `stations-summary`, as the function `main` in
`stations.summary`. The install wrote a script by that name into the environment's `bin` folder,
which calls `main`. Here it runs from `scratch`, on a file of readings there:


In [16]:
(SCRATCH / "tuesday.csv").write_text("Bergen,4.2\nBergen,5.8\nBergen,\nOslo,-2.4\nOslo,-1.6\nSvalbard,\n", encoding="utf-8")

code, printed = run("stations/.venv/bin/stations-summary", "tuesday.csv")
print("exit code:", code)
print(printed)


exit code: 0
Bergen 5.0 C, 41.0 F
Oslo -2.0 C, 28.4 F
Svalbard no readings


`main(arguments=None)` reads the command's arguments from `sys.argv` when it is called with none,
which is how the script calls it, and a test can still call `main(["tuesday.csv"])` with arguments
of its own. The script's first line names the environment's Python, as every script pip installs
does, so the command runs in the environment without anything activated.

### A wheel, and a regular install of it

A **wheel** is the file pip installs from: a zip archive of the package and its metadata, named for
the project, its version, and the Pythons and systems it runs on. `pip wheel` builds one:


In [17]:
code, printed = run(sys.executable, "-m", "pip", "wheel", "--disable-pip-version-check", "-q", "--no-deps",
                    "-w", "dist", ".", folder=PROJECT)
print("build exit code:", code)

wheels = sorted((PROJECT / "dist").glob("*.whl"))
print("built:", [wheel.name for wheel in wheels])
with zipfile.ZipFile(wheels[0]) as wheel:
    names = wheel.namelist()
print("package files:", [name for name in names if not name.startswith("stations-0.1.0.dist-info/")])
print("metadata:", [name.split("/")[-1] for name in names if name.split("/")[-1] in ("METADATA", "entry_points.txt")])


build exit code: 0
built: ['stations-0.1.0-py3-none-any.whl']
package files: ['stations/__init__.py', 'stations/readings.py', 'stations/summary.py']
metadata: ['METADATA', 'entry_points.txt']


`stations-0.1.0-py3-none-any.whl` says: the project `stations`, version 0.1.0, for any Python 3,
with no compiled code, on any system. Inside are the package's three files and the metadata that pip
reads, among them `entry_points.txt`, which holds the command. A user installs the wheel, or the
same project from the Python Package Index, and gets a copy:


In [18]:
run(sys.executable, "-m", "venv", "--without-pip", "user-env")
code, printed = pip("user-env", "install", "-q", PROJECT.resolve() / "dist" / wheels[0].name)
print("install exit code:", code)

check = ("import stations.readings as readings, sysconfig; "
         "print('in site-packages:', readings.__file__.startswith(sysconfig.get_paths()['purelib']))")
print(run("user-env/bin/python", "-c", check)[1])
print(run("user-env/bin/python", "-c", "from stations.readings import to_kelvin; print(to_kelvin(100))")[1])


install exit code: 0
in site-packages: True
373.15


The regular install put a copy of the package into `user-env`'s `site-packages`, with `to_kelvin`,
which was in the files when the wheel was built. The build also left folders in the project that
belong to no repository: `dist`, `build`, and `src/stations.egg-info`, which the capstone ignores.

### The project rebuilt from its files in another folder

The pieces of this notebook, in one move. A `.gitignore` names what the project makes and never
keeps. A copy of the project that leaves out the same things stands in for a fresh checkout on
another computer, which installs the project in editable mode, runs its tests from outside the
project, and runs its command:


In [19]:
%%writefile scratch/stations/.gitignore
.venv/
build/
dist/
*.egg-info/
.pytest_cache/


Writing scratch/stations/.gitignore


In [20]:
elsewhere = SCRATCH / "elsewhere"
shutil.rmtree(elsewhere, ignore_errors=True)
shutil.copytree(PROJECT, elsewhere, ignore=shutil.ignore_patterns(".venv", "build", "dist", "*.egg-info", ".pytest_cache"))
print("copied:", sorted(str(path.relative_to(elsewhere)) for path in elsewhere.rglob("*") if path.is_file()))

steps = [("make the environment", [sys.executable, "-m", "venv", "--without-pip", "elsewhere/.venv"], SCRATCH),
         ("install the project, editable", [sys.executable, "-m", "pip", "--python", ".venv", "--disable-pip-version-check",
                                            "install", "-q", "-e", ".[test]"], elsewhere),
         ("run the tests from outside", ["elsewhere/.venv/bin/pytest", "-q", "--no-header", "elsewhere"], SCRATCH),
         ("run the command", ["elsewhere/.venv/bin/stations-summary", "tuesday.csv"], SCRATCH)]
for name, command, folder in steps:
    code, printed = run(*command, folder=folder)
    print(f"--- {name}: exit code {code}")
    if printed:
        print(printed)


copied: ['.gitignore', 'pyproject.toml', 'src/stations/__init__.py', 'src/stations/readings.py', 'src/stations/summary.py', 'tests/test_readings.py']
--- make the environment: exit code 0
--- install the project, editable: exit code 0
--- run the tests from outside: exit code 0
...                                                                      [100%]
3 passed
--- run the command: exit code 0
Bergen 5.0 C, 41.0 F
Oslo -2.0 C, 28.4 F
Svalbard no readings


The copy held six files: the `.gitignore`, `pyproject.toml`, the package's three files and the tests.
From those, one install gave an environment in which the tests pass from outside the project and the
command runs, with no folder added to any path by hand.

### Where each part came from

| In the project | What it relies on | The section that showed it |
|---|---|---|
| `src/stations/` | a package that imports only once it is installed | The src layout, and a package that imports only once it is installed |
| `pyproject.toml` | the project's name, version, needs and commands, in one record | pyproject.toml, read as a record of the project |
| `pip install -e ".[test]"` | an editable install, with the test extra | An editable install, and imports that work from any folder |
| `pytest` from outside the project | tests that import the installed package | Tests that import the installed package |
| `stations-summary` | `[project.scripts]`, turned into a script by the install | A command of its own |
| `.gitignore` | `build`, `dist` and `.egg-info`, which a build makes | A wheel, and a regular install of it |

The project no longer depends on the folder a command runs in: it depends on being installed, which
one command does, on any computer.


## Your turn

Six tasks. Write your answer in the cell under each task and run it.

Try a task before you look at its answer. Reading a solution teaches you much less than getting
there yourself, even slowly.

When you are ready: [**open the solutions notebook**](https://colab.research.google.com/github/johnfisher-ai/Python-Visual-Guides/blob/main/notebooks/testing-and-packaging/09-project-layout-solutions.ipynb).

**1.** Print the files of the project in `scratch/stations`, as paths relative to the project,
leaving out what the tools made: everything under `.venv`, `.pytest_cache`, `build`, `dist` and
`src/stations.egg-info`.


In [21]:
# your code here


**2.** Read `pyproject.toml` with `tomllib`, and print the project's name and version, and the
function its command calls.


In [22]:
# your code here


**3.** From `scratch`, run the project's environment's Python to import `stations.readings` and print
`to_fahrenheit(-40)`.


In [23]:
# your code here


**4.** Add a function, `to_celsius(fahrenheit)`, to `src/stations/readings.py`, and call it from
`scratch` with `212`, without installing again.


In [24]:
# your code here


**5.** Add a test of `to_celsius` to `tests/test_readings.py`, and run the tests with the
environment's `pytest` script from `scratch`.


In [25]:
# your code here


**6.** Build a wheel of the project into `scratch/task-dist`, and print the names of the package's
files inside it, leaving out the metadata.


In [26]:
# your code here


## Common errors

### error: Multiple top-level modules discovered in a flat-layout


In [27]:
(FLAT / "pyproject.toml").write_text((PROJECT / "pyproject.toml").read_text())

code, printed = pip(".venv", "install", "-e", ".", folder=FLAT)
print("exit code:", code)
print(next(line for line in printed.splitlines() if "Multiple top-level" in line).strip().rsplit(":", 1)[0])


exit code: 1
error: Multiple top-level modules discovered in a flat-layout


setuptools looked for what to package in the flat project's folder, and found two modules at the
top, `readings.py` and `summary.py`, with nothing to say which of them, or whether both, make up the
project. It refuses to guess, and pip installs nothing. The full message lists the modules, in an
order that differs between computers, so the cell prints its first part. A package under `src`
settles the question, which is what the src layout is for:


In [28]:
code, printed = pip(".venv", "install", "-q", "-e", ".", folder=PROJECT)
print("exit code:", code)


exit code: 0


### ModuleNotFoundError: No module named 'stations'


In [29]:
code, printed = run(sys.executable, "-c", "import stations.readings", folder=PROJECT)

print("exit code:", code)
print(printed.splitlines()[-1])


exit code: 1
ModuleNotFoundError: No module named 'stations'


The notebook's own Python ran the import, in the project's folder, and the project was installed
into `stations/.venv`, not into the notebook's Python. An install belongs to one environment, and a
program run by another Python does not see it. Run the Python of the environment the project was
installed into:


In [30]:
code, printed = run(".venv/bin/python", "-c", "import stations.readings; print('imported')", folder=PROJECT)

print("exit code:", code, "|", printed)


exit code: 0 | imported


### ModuleNotFoundError: No module named 'readings'


In [31]:
summary = PROJECT / "src" / "stations" / "summary.py"
source = summary.read_text()
summary.write_text(source.replace("from stations.readings import", "from readings import"))

code, printed = run("stations/.venv/bin/stations-summary", "tuesday.csv")
print("exit code:", code)
print(printed.splitlines()[-1])


exit code: 1
ModuleNotFoundError: No module named 'readings'


`from readings import` worked in the flat project, where `readings.py` sat in the folder the command
ran in. Inside a package, `readings` is `stations.readings`, and no module called `readings` is on
the import path. A module in a package imports its neighbors by the package's name:


In [32]:
summary.write_text(source)

code, printed = run("stations/.venv/bin/stations-summary", "tuesday.csv")
print("exit code:", code)
print(printed)


exit code: 0
Bergen 5.0 C, 41.0 F
Oslo -2.0 C, 28.4 F
Svalbard no readings


`from .readings import`, with a dot for the package the module is in, works too, and says the same
thing.

### No error, and yesterday's code: a regular install that does not see an edit


In [33]:
readings_file = PROJECT / "src" / "stations" / "readings.py"
readings_file.write_text(readings_file.read_text().replace("return celsius + 273.15", "return round(celsius + 273.15, 1)"))

check = "from stations.readings import to_kelvin; print(to_kelvin(21.37))"
for environment in ["stations/.venv", "user-env"]:
    code, printed = run(f"{environment}/bin/python", "-c", check)
    print(f"{environment:<15} {printed}")


stations/.venv  294.5
user-env        294.52


The edit rounds kelvin to one decimal. The editable install in `stations/.venv` reads the project's
files, so it rounds. `user-env` got a regular install, a copy made when the wheel was built, so it
still runs yesterday's code, and says nothing about it. A regular install sees an edit only when the
project is built and installed again:


In [34]:
code, printed = run(sys.executable, "-m", "pip", "wheel", "--disable-pip-version-check", "-q", "--no-deps",
                    "-w", "dist", ".", folder=PROJECT)
print("build exit code:", code)
code, printed = pip("user-env", "install", "-q", "--force-reinstall", PROJECT.resolve() / "dist" / wheels[0].name)
print("install exit code:", code)

print("user-env       ", run("user-env/bin/python", "-c", check)[1])


build exit code: 0
install exit code: 0
user-env        294.5


`--force-reinstall` makes pip install the wheel again although the version, 0.1.0, has not changed.
A project that others install raises its version with every release, so that pip knows a new release
is new.

Last, the notebook is finished with its files, so this cell removes the scratch folder, with both
projects, every environment, and the wheel:


In [35]:
shutil.rmtree("scratch")

print("scratch still there:", Path("scratch").exists())


scratch still there: False


## Recap

- In the src layout, the package sits in `src/stations/`, the tests in `tests/`, and nothing imports
  the package until it is installed.
- `pyproject.toml` names the build backend in `[build-system]`, and the project's name, version,
  Pythons, dependencies and commands in `[project]`, and pytest reads its settings from it too.
- `pip install -e .` installs a link to the project's files, so an edit takes effect at once, and
  `pip install .` or a wheel installs a copy.
- An installed package imports from any folder, with `pytest` or `python -m pytest`, which is the
  reason to install a project rather than lean on the folder a command runs in.
- `[project.scripts]` turns a function into a command in the environment's `bin` folder.
- `pip wheel` builds the file a user installs, and a `.gitignore` keeps `build`, `dist`, `.egg-info`
  and the environment out of the repository.
- A module in a package imports its neighbors by the package's name: `from stations.readings import`.


## What is next

The **Continuous Integration** notebook runs this project's tests on a server for every push: a
workflow file that checks the project out, installs it with `pip install -e .`, runs pytest, and
marks the run failed when a test fails.


---

&#8592; **Previous:** [Requirements and Pinning](https://colab.research.google.com/github/johnfisher-ai/Python-Visual-Guides/blob/main/notebooks/testing-and-packaging/08-requirements-and-pinning.ipynb)  &nbsp;·&nbsp;  [Testing and Packaging Notebooks](https://johnfisher-ai.github.io/Python-Visual-Guides/testing-and-packaging.html)  &nbsp;·&nbsp;  **Next:** [Continuous Integration](https://colab.research.google.com/github/johnfisher-ai/Python-Visual-Guides/blob/main/notebooks/testing-and-packaging/10-continuous-integration.ipynb) &#8594;
